# Hands-on: the full pass on an export nobody has seen

Anand's analyst sends a second export, a 97-row slice, with a note that says only: "this one came
through a different route."

It has defects. They are **not** the same defects as the class file, so nothing from the session
can be pasted across. Run the full pass: profile, decide, record, reconcile.

Fill every `__TODO__`. **Post the five letters** from the last section as one string.

In [ ]:
import collections
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

rows = kit.load_csv("C2_W01_D03_takehome_STUDENT.csv")
print(f"{len(rows)} rows read")
print(rows[0])

## 1. Profile first, and do not fix anything yet

Three counts per field. Anything that disagrees with what you expected is a finding.

In [ ]:
kit.flow(["__TODO__", "__TODO__", "__TODO__"], lit=2,
         title="The three counts a profile reports per field")

In [ ]:
def profile(records, field):
    present = [r for r in records if r.get(field) not in (None, "")]
    convertible = 0
    for r in present:
        try:
            int(r[field])
            convertible += 1
        except (TypeError, ValueError):
            pass
    return len(present), convertible, len({r[field] for r in present})


for field in ("order_id", "amount", "status", "order_date"):
    print(field, profile(rows, field))

In [ ]:
kit.check("the row count is not the id count", len({r["order_id"] for r in rows}) < len(rows),
          f"{len({r['order_id'] for r in rows})} ids in {len(rows)} rows")

## 2. Name every defect before you decide anything

There are **four kinds** in this file. Find them and fill the table. One of them does not exist in
the class file at all.

In [ ]:
repeated = __TODO__          # how many order ids appear more than once
non_numeric = __TODO__       # rows whose amount will not convert to an int
negative = __TODO__          # rows whose amount converts and is below zero
blank_status = __TODO__      # rows with no status

kit.table(["defect", "rows"],
          [("repeated order id", repeated), ("amount will not convert", non_numeric),
           ("amount is negative", negative), ("status is blank", blank_status)],
          caption="What is wrong with this export")

In [ ]:
kit.check("you found the repeated ids", repeated > 0, f"{repeated}")
kit.check("you found the row that is not an order at all", non_numeric >= 1, f"{non_numeric}")
kit.check("you found the negative amount", negative == 1, f"{negative}")

## 3. The one that is not a defect

One row in this file has an `order_date` in a different format from every other row. It is not
corrupt and it is not a duplicate.

Find it, and decide in the comment below whether it belongs in clean or in rejected, **and why**.

In [ ]:
odd_dates = [r for r in rows if __TODO__]
print(odd_dates)
# My decision, and the reason: __TODO__

## 3b. Which fields decide that two rows are the same order?

Fill the three branches of the identity rule for this file, and mark the branch you actually took.

In [ ]:
kit.tree(
    {"label": "two rows match",
     "branches": [
         ("__TODO__", {"label": "__TODO__"}),
         ("__TODO__", {"label": "__TODO__"}),
         ("__TODO__", {"label": "__TODO__"}),
     ]},
    taken=["__TODO__"], title="The identity rule, as you applied it here")

## 4. The pass, with a reason on every rejection

In [ ]:
def amount(r, default=None):
    try:
        return int(r["amount"])
    except (TypeError, ValueError):
        return default


clean, rejected, seen = [], [], set()
for r in rows:
    if r["order_id"] in seen:
        rejected.append((r, "__TODO__"))
        continue
    seen.add(r["order_id"])
    if __TODO__:
        rejected.append((r, "__TODO__"))
        continue
    clean.append(r)

reasons = collections.Counter(reason for _, reason in rejected)
kit.table(["reason", "rows"], sorted(reasons.items()), caption="Your rejects log")

In [ ]:
kit.check("input equals clean plus rejected",
          len(clean) + len(rejected) == len(rows),
          f"{len(clean)} + {len(rejected)} against {len(rows)}")
kit.check("every rejection carries a reason", all(reason.strip() for _, reason in rejected),
          "a reason with no words in it is not a reason")

## 5. The bridge

Draw the steps from the exported total to your reconciled one, with the number at each step.

In [ ]:
raw_total = sum(amount(r) or 0 for r in rows)
clean_total = sum(amount(r) for r in clean)
kit.vflow(["__TODO__", "__TODO__", "__TODO__", "__TODO__"], lit=3,
          title="From the export to the number you would sign")
print(f"exported Rs {raw_total:,}   reconciled Rs {clean_total:,}   "
      f"gap Rs {raw_total - clean_total:,}")

## 6. The five letters to post

**Q1.** A row's `order_id` is the text `order_id`. What is it?
`a` a corrupt order · `b` a header line read as a record ·
`c` an order placed by an internal system · `d` a duplicate of the first row

**Q2.** An amount of `-2400` converts to an integer without error. What is your move?
`a` reject it, because an order cannot be negative ·
`b` default it to zero, because the sign is obviously a typo ·
`c` keep it and flag it, because it is probably a refund posted as an order ·
`d` take the absolute value, since the magnitude is the real figure

**Q3.** Two rows share an id and every other field is identical. Which check finds it?
`a` only a check on the whole record · `b` only a check on the id ·
`c` either, because both describe the same two rows · `d` neither, they are two real orders

**Q4.** Your clean total is higher than the exported total. What happened?
`a` you dropped a negative amount, so removing it raised the sum ·
`b` you double counted a row during the pass · `c` a conversion defaulted to a large number ·
`d` that cannot happen if the pass is correct

**Q5.** The date in the second format. Does it belong in clean or rejected?
`a` rejected, because a format difference means the row is untrustworthy ·
`b` clean, once parsed, and the format noted in the log ·
`c` rejected, because you cannot prove which of day and month is which ·
`d` clean, and the difference ignored since the date is not used today

In [ ]:
my_answers = "__TODO__"
kit.check("five letters posted", len(my_answers) == 5 and my_answers.isalpha(),
          f"got {my_answers!r}")
kit.check_summary()

## 7. The note

Four sentences to Anand's analyst. What arrived, what you rejected and why, what number you would
sign, and the one thing you had to make a judgment about.

> __TODO__